<a href="https://colab.research.google.com/github/zazotona301-oss/zazo/blob/main/zazo_train_upload_huggingface_(2).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Zazo Brain — تدريب LoRA ثم رفعه إلى Hugging Face

هذا الدفتر يدرب محول LoRA لنموذج `google/gemma-2b-it`، ثم يرفع ملفي المحول إلى مستودع Hugging Face. بعد الرفع يمكنه اختيارياً تشغيل GitHub Action الموجود في مستودع Zazo ليقوم بالمسار التالي:

`Hugging Face → GitHub Action → Cloudflare Workers AI → active-lora في KV`

لا تضع أي Token داخل الدفتر. استخدم Colab Secrets مع تفعيل **Notebook access**. يلزم GPU في Colab، ويلزم توكن Hugging Face بصلاحية كتابة على مستودع المحول، مع قبول شروط Gemma أولاً.

## 1. اختيار ملف التدريب من الهاتف

شغّل الخلية التالية، ثم اضغط **Choose Files / اختيار الملفات** لفتح مجلدات الهاتف. يقبل الدفتر ملف `OnlineRetail.xlsx` مباشرة، أو JSONL/JSON/CSV/TSV.


In [ ]:
from pathlib import Path
from google.colab import files

SUPPORTED = {'.xlsx', '.xls', '.jsonl', '.ndjson', '.json', '.csv', '.tsv'}
print('اختر ملف OnlineRetail.xlsx من الهاتف...')
uploaded = files.upload()
if not uploaded:
    raise RuntimeError('لم يتم اختيار ملف التدريب من الهاتف.')
name, content = next(iter(uploaded.items()))
TRAIN_FILE = Path('/content') / Path(name).name
TRAIN_FILE.write_bytes(content)
if TRAIN_FILE.suffix.lower() not in SUPPORTED:
    raise RuntimeError(f'صيغة غير مدعومة: {TRAIN_FILE.suffix}. اختر XLSX أو JSONL أو JSON أو CSV أو TSV.')
print('تم رفع الملف:', TRAIN_FILE)

اختر ملف OnlineRetail.xlsx من الهاتف...


Saving pricerunner_aggregate.csv to pricerunner_aggregate (1).csv
تم رفع الملف: /content/pricerunner_aggregate (1).csv


In [ ]:

import json

# تأكد من أن مسار الملف صحيح
TRAIN_FILE = "مسار_ملف_التدريب_هنا"

try:
    with open(TRAIN_FILE, 'r', encoding='utf-8') as f:
        # قراءة أول 5 أسطر أو عينة للتأكد من محتوى الملف
        lines = [f.readline() for _ in range(5)]

    print("تم قراءة الملف بنجاح. إليك عينة من المحتوى:")
    for i, line in enumerate(lines):
        if line.strip():
            print(f"سطر {i+1}:", json.loads(line) if TRAIN_FILE.endswith('.jsonl') else line.strip())

except Exception as e:
    print(f"حدث خطأ أثناء قراءة الملف: {e}")

حدث خطأ أثناء قراءة الملف: [Errno 2] No such file or directory: 'مسار_ملف_التدريب_هنا'


## 1. فحص الموارد المتاحة في جلسة Colab

هذه الخلية توضح الموارد التي خصصتها Google لجلسة Runtime الحالية. لا يمكن للدفتر معرفة رصيد GPU على الحساب قبل تشغيل جلسة؛ اضغط **Connect** أو اختر GPU من **Runtime → Change runtime type** ثم شغّلها.

In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path
try:
    import psutil
except ImportError:
    psutil = None

def gib(value):
    return round(value / (1024 ** 3), 2)

print('Python:', sys.version.split()[0])
print('Colab:', 'نعم' if 'google.colab' in sys.modules or Path('/content').exists() else 'غير مؤكد')

try:
    import torch
    gpu_count = torch.cuda.device_count()
    if not torch.cuda.is_available() or gpu_count == 0:
        print('GPU: غير متاح في جلسة Runtime الحالية')
        print('الحل: Runtime → Change runtime type → اختر T4 GPU أو GPU متاح → Save ثم أعد تشغيل الخلية.')
    else:
        print(f'GPU: متاح ({gpu_count} جهاز)')
        for index in range(gpu_count):
            props = torch.cuda.get_device_properties(index)
            free, total = torch.cuda.mem_get_info(index)
            print(f'  GPU {index}: {props.name}')
            print(f'  الذاكرة: {gib(total)} GiB إجمالي، {gib(free)} GiB متاح')
        try:
            smi = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,memory.free,utilization.gpu', '--format=csv,noheader,nounits'], capture_output=True, text=True, check=False)
            if smi.stdout.strip(): print('nvidia-smi:\n' + smi.stdout.strip())
        except FileNotFoundError:
            pass
except Exception as error:
    print('تعذر فحص GPU:', error)

if psutil:
    memory = psutil.virtual_memory()
    print(f'RAM: {gib(memory.total)} GiB إجمالي، {gib(memory.available)} GiB متاح')
else:
    print('RAM: تعذر الفحص؛ psutil غير مثبتة بعد')
disk = shutil.disk_usage('/content')
print(f'مساحة /content: {gib(disk.total)} GiB إجمالي، {gib(disk.free)} GiB متاح')

if 'torch' in globals() and not torch.cuda.is_available():
    raise RuntimeError('أوقفنا الدفتر لأن GPU غير متاح. فعّل GPU ثم أعد تشغيل الخلية قبل التدريب.')

Python: 3.13.16
Colab: نعم
GPU: متاح (1 جهاز)
  GPU 0: Tesla T4
  الذاكرة: 14.56 GiB إجمالي، 14.46 GiB متاح
nvidia-smi:
Tesla T4, 15360, 14808, 0
RAM: 12.67 GiB إجمالي، 9.36 GiB متاح
مساحة /content: 112.64 GiB إجمالي، 69.44 GiB متاح


## 1. تثبيت الحزم

In [ ]:
%pip install -q -U transformers datasets peft accelerate bitsandbytes huggingface_hub safetensors requests pandas openpyxl
%pip uninstall -y torchao
print('تم تثبيت الحزم وإزالة تعارض torchao')

تم تثبيت الحزم وإزالة تعارض torchao


## 2. الإعداد والتحقق من GPU

In [ ]:
from pathlib import Path
import os, json, csv, shutil, zipfile, getpass, re
import torch

if not torch.cuda.is_available():
    raise RuntimeError('فعّل GPU من Runtime → Change runtime type → T4 GPU أو أي GPU متاح قبل التدريب.')
print('GPU:', torch.cuda.get_device_name(0))

MODEL_NAME = 'google/gemma-2b-it'
HF_REPO_ID = 'zazo300/zazo-gemma-lora'
HF_REVISION = 'main'
MAX_LENGTH = 512
EPOCHS = 1
BATCH_SIZE = 1
GRADIENT_ACCUMULATION = 8
OUTPUT_DIR = Path('/content/zazo_lora')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('مستودع Hugging Face:', HF_REPO_ID)

GPU: Tesla T4
مستودع Hugging Face: zazo300/zazo-gemma-lora


## 4. تسجيل الدخول إلى Hugging Face

أنشئ Secret باسم `HF_TOKEN` بصلاحية **Write** على الأقل. لا تكتب قيمة التوكن داخل الخلية.

In [ ]:

import pandas as pd

# مسار الملف الفعلي الذي تم رفعه
TRAIN_FILE = "/content/pricerunner_aggregate (1).csv"

try:
    # قراءة الملف باستخدام pandas (مع تحديد الفاصل التلقائي)
    df = pd.read_csv(TRAIN_FILE, sep=None, engine='python')

    print("تم تحميل الملف بنجاح!")
    print(f"عدد السطور: {df.shape[0]}, عدد الأعمدة: {df.shape[1]}")
    print("\nعينة من البيانات (أول 5 أسطر):")
    display(df.head())

except Exception as e:
    print(f"حدث خطأ أثناء قراءة الملف: {e}")

تم تحميل الملف بنجاح!
عدد السطور: 35311, عدد الأعمدة: 7

عينة من البيانات (أول 5 أسطر):


,Product ID,Product Title,Merchant ID,Cluster ID,Cluster Label,Category ID,Category Label
0,1,apple iphone 8 plus 64gb silver,1,1,Apple iPhone 8 Plus 64GB,2612,Mobile Phones
1,2,apple iphone 8 plus 64 gb spacegrau,2,1,Apple iPhone 8 Plus 64GB,2612,Mobile Phones
2,3,apple mq8n2b/a iphone 8 plus 64gb 5.5 12mp sim...,3,1,Apple iPhone 8 Plus 64GB,2612,Mobile Phones
3,4,apple iphone 8 plus 64gb space grey,4,1,Apple iPhone 8 Plus 64GB,2612,Mobile Phones
4,5,apple iphone 8 plus gold 5.5 64gb 4g unlocked ...,5,1,Apple iPhone 8 Plus 64GB,2612,Mobile Phones


In [ ]:
from huggingface_hub import login, HfApi
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = None
if not HF_TOKEN:
    HF_TOKEN = getpass.getpass('HF_TOKEN من Hugging Face: ').strip()
if not HF_TOKEN.startswith('hf_'):
    raise RuntimeError('توكن Hugging Face غير صالح.')
login(token=HF_TOKEN, add_to_git_credential=False)
api = HfApi(token=HF_TOKEN)
api.create_repo(repo_id=HF_REPO_ID, repo_type='model', exist_ok=True, private=False)
print('تم التحقق من مستودع Hugging Face:', HF_REPO_ID)

HF_TOKEN من Hugging Face: ··········
تم التحقق من مستودع Hugging Face: zazo300/zazo-gemma-lora


## 5. تدريب Gemma 2B باستخدام LoRA

In [ ]:

import pandas as pd
from datasets import Dataset
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

# 1. قراءة ملف الـ CSV الذي قمت برفعه
TRAIN_FILE = "/content/pricerunner_aggregate (1).csv"
df = pd.read_csv(TRAIN_FILE, sep=None, engine='python')

# 2. تحويل الأعمدة إلى حقل نصي واحد يتناسب مع التدريب (مثلاً دمج العنوان والفئة)
# تأكد من أن أسماء الأعمدة مطابقة لما لديك، أو سنستخدم الأعمدة الموجودة
df['text'] = "اسم المنتج: " + df.iloc[:, 0].astype(str) + " | الفئة: " + df.iloc[:, 1].astype(str)

# إنشاء الـ dataset من الـ DataFrame
records = df[['text']].to_dict(orient='records')
dataset = Dataset.from_list(records)

print("تم تجهيز عدد أمثلة التدريب:", len(dataset))

# 3. دالة الترميز
MAX_LENGTH = 512
def tokenize(batch):
    return tokenizer(batch['text'], truncation=True, max_length=MAX_LENGTH)

# تطبيق الترميز
tokenized = dataset.map(tokenize, batched=True, remove_columns=dataset.column_names)

# 4. إعدادات التدريب
args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=1,
    save_strategy="no",
    fp16=True,
    report_to="none",
    optim="paged_adamw_8bit"
)

# 5. بناء المدرب (Trainer) وبدء التدريب
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False)
)

print("جاري بدء التدريب الآن...")
trainer.train()

تم تجهيز عدد أمثلة التدريب: 35311


Map:   0%|          | 0/35311 [00:00<?, ? examples/s]

جاري بدء التدريب الآن...


Step,Training Loss
1,9.299999
2,8.698969
3,9.042405
4,9.048544
5,8.356627
6,8.172277
7,7.856468
8,8.322654
9,7.333867
10,6.525027


TrainOutput(global_step=4414, training_loss=2.943636349385067, metrics={'train_runtime': 2236.7144, 'train_samples_per_second': 15.787, 'train_steps_per_second': 1.973, 'total_flos': 1.5803865952223232e+16, 'train_loss': 2.943636349385067, 'epoch': 1.0})

## 6. تنزيل ناتج LoRA إلى الهاتف أو رفعه اختيارياً

الوضع الافتراضي ينزّل ملف ZIP إلى الهاتف. بعد ذلك يمكنك تسجيل الدخول إلى حساب Hugging Face الأصلي ورفع الملفين يدوياً. غيّر `AUTO_UPLOAD_TO_HF` إلى `True` فقط إذا أردت الرفع من داخل Colab.


In [ ]:
import zipfile
from google.colab import files

AUTO_UPLOAD_TO_HF = False
config_path = OUTPUT_DIR / 'adapter_config.json'
weights_path = OUTPUT_DIR / 'adapter_model.safetensors'
if not config_path.exists() or not weights_path.exists():
    raise FileNotFoundError('لم ينتج التدريب adapter_config.json و adapter_model.safetensors.')
config = json.loads(config_path.read_text())
config.setdefault('model_type', 'gemma')
config_path.write_text(json.dumps(config, ensure_ascii=False, indent=2), encoding='utf-8')
manifest = {'base_model': MODEL_NAME, 'examples': len(dataset), 'source_file': TRAIN_FILE.name, 'workflow': 'sync-huggingface-cloudflare.yml'}
manifest_path = OUTPUT_DIR / 'training_manifest.json'
manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')
zip_path = OUTPUT_DIR.parent / 'zazo_lora_for_huggingface.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as archive:
    archive.write(config_path, 'adapter_config.json')
    archive.write(weights_path, 'adapter_model.safetensors')
    archive.write(manifest_path, 'training_manifest.json')
print('تم تجهيز ملف التنزيل:', zip_path)
files.download(str(zip_path))

if AUTO_UPLOAD_TO_HF:
    api.upload_file(path_or_fileobj=str(config_path), path_in_repo='adapter_config (1).json', repo_id=HF_REPO_ID, repo_type='model', revision=HF_REVISION, commit_message='Upload Zazo LoRA adapter config')
    api.upload_file(path_or_fileobj=str(weights_path), path_in_repo='adapter_model (1).safetensors', repo_id=HF_REPO_ID, repo_type='model', revision=HF_REVISION, commit_message='Upload Zazo LoRA adapter weights')
    api.upload_file(path_or_fileobj=str(manifest_path), path_in_repo='training_manifest.json', repo_id=HF_REPO_ID, repo_type='model', revision=HF_REVISION, commit_message='Upload training manifest')
    print('تم الرفع إلى Hugging Face:', f'https://huggingface.co/{HF_REPO_ID}')
else:
    print('وضع التنزيل فقط: ارفع محتويات ZIP يدوياً إلى حساب Hugging Face الأصلي.')

## 7. تشغيل GitHub Action اختيارياً

أنشئ Secret في Colab باسم `GH_TOKEN` بصلاحية Actions: write على مستودع GitHub. إذا لم تضفه، شغّل الـworkflow يدوياً من GitHub بعد انتهاء الخلية.

In [ ]:
import requests
GITHUB_REPO = 'zazotona301-oss/zazo'
try:
    from google.colab import userdata
    GH_TOKEN = userdata.get('GH_TOKEN')
except Exception:
    GH_TOKEN = None

workflow_url = f'https://github.com/{GITHUB_REPO}/actions/workflows/sync-huggingface-cloudflare.yml'
if not GH_TOKEN:
    print('لم يتم تشغيل GitHub Action تلقائياً.')
    print('شغّله يدوياً من:', workflow_url)
else:
    response = requests.post(f'https://api.github.com/repos/{GITHUB_REPO}/actions/workflows/sync-huggingface-cloudflare.yml/dispatches', headers={'Accept': 'application/vnd.github+json', 'Authorization': f'Bearer {GH_TOKEN}', 'X-GitHub-Api-Version': '2022-11-28'}, json={'ref': 'main'}, timeout=30)
    if response.status_code != 204:
        raise RuntimeError(f'فشل تشغيل GitHub Action: HTTP {response.status_code} {response.text[:500]}')
    print('تم تشغيل GitHub Action بنجاح.')
    print('تابع النتيجة من:', workflow_url)

## النتيجة المتوقعة

بعد نجاح الـworkflow سيقوم GitHub بتحميل الملفين من Hugging Face، إنشاء fine-tune جديد في Cloudflare Workers AI، رفع الأوزان، ثم تحديث مفتاح `active-lora` في KV. لا تضع `CF_AI_TOKEN` أو `CF_KV_TOKEN` داخل Colab؛ هذه الأسرار تبقى في GitHub Actions Secrets كما هي حالياً.